# Audio Denoising

Estimate noise from low-energy frames, suppress tonal components with notch filters, and apply spectral subtraction.

Run the cells in order. This notebook is independent of the other notebooks.

The original STFT/ISTFT scaling and boundary handling are retained; the exported audio is slightly shorter than the input.


In [ ]:
from pathlib import Path

# Resolve paths when launched from the repository root or notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "audio"
OUTPUT_DIR = ROOT / "outputs"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Waveform and spectrogram analysis


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Dict, Optional
from scipy.io import wavfile
from scipy.signal import spectrogram


# ---------- I/O & preprocessing ----------
def read_wav_mono(path: Path) -> Tuple[int, np.ndarray]:
    """Read WAV and return (fs, mono float32 in [-1, 1])."""
    fs, x = wavfile.read(str(path))
    return fs, to_mono_float(x)


def to_mono_float(x: np.ndarray) -> np.ndarray:
    """int/float -> float32 in [-1,1]; stereo -> mono (mean)."""
    if x.ndim == 2:
        x = x.mean(axis=1)
    if x.dtype.kind in "iu":  # int/uint
        x = x.astype(np.float32) / np.iinfo(x.dtype).max
    else:
        x = x.astype(np.float32)
    return x


# ---------- analysis ----------
def compute_log_spectrogram(
    y: np.ndarray,
    fs: int,
    *,
    n_fft: int = 4096,
    hop: int = 1024,
    window: str = "hann",
    db_floor: float = -120.0
) -> Dict[str, np.ndarray]:
    """
    Compute magnitude spectrogram and return log-magnitude (dB) with a stable floor.
    Returns dict with f (Hz), t (s), mag (linear), and mag_db (dB).
    """
    f, t, S = spectrogram(
        y, fs=fs, nperseg=n_fft, noverlap=n_fft - hop,
        window=window, detrend=False, mode="magnitude"
    )
    # Avoid log(0) and clamp to floor
    eps = 1e-12
    mag_db = 20.0 * np.log10(S + eps)
    mag_db = np.maximum(mag_db, db_floor)
    return {"f": f, "t": t, "mag": S, "mag_db": mag_db}


# ---------- plotting ----------
def plot_waveform(y: np.ndarray, fs: int, title: str = "Waveform") -> None:
    t = np.arange(len(y), dtype=float) / fs
    plt.figure(figsize=(10, 3))
    plt.plot(t, y, color = "green")
    plt.title(title)
    plt.xlabel("time (s)")
    plt.ylabel("amplitude")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_spectrogram_db(
    f: np.ndarray, t: np.ndarray, mag_db: np.ndarray,
    *, title: str = "Log-magnitude spectrogram (dB)", vmin: Optional[float] = None
) -> None:
    plt.figure(figsize=(10, 4))
    pcm = plt.pcolormesh(t, f, mag_db, shading="gouraud", vmin=vmin)
    plt.title(title)
    plt.xlabel("time (s)")
    plt.ylabel("frequency (Hz)")
    plt.colorbar(pcm, label="dB")
    plt.tight_layout()
    plt.show()

wav_path = Path((DATA_DIR / "noisy1.wav"))
fs, noisy = read_wav_mono(wav_path)
print(f"length = {len(noisy)/fs:.2f} s | fs = {fs} Hz | Number of Samples = {len(noisy)}")

NFFT, HOP = 4096, 1024
spec = compute_log_spectrogram(noisy, fs, n_fft=NFFT, hop=HOP, window="hann", db_floor=-120)

plot_waveform(noisy, fs, title=f"{wav_path.name} — waveform")

In [ ]:
plot_spectrogram_db(spec["f"], spec["t"], spec["mag_db"], title="spectrogram log-magnitude")

## Noise estimation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram

# --- helpers (same as before) ---
def compute_frame_energy(Sxx, *, power=True):
    return (Sxx**2).mean(axis=0) if power else Sxx.mean(axis=0)

def gap_mask_from_percentile(values, perc):
    thr = np.percentile(values, perc)
    return (values <= thr), float(thr)

def estimate_noise_from_gaps(Sxx, gap_mask, eps=1e-12):
    avg_mag = Sxx[:, gap_mask].mean(axis=1) if gap_mask.sum() > 0 else Sxx.mean(axis=1)
    mag_db = 20*np.log10(np.maximum(avg_mag, eps))
    return avg_mag, mag_db

# --- full workflow example ---
fs, noisy = wavfile.read((DATA_DIR / "noisy1.wav"))
noisy = noisy.astype(np.float32) / np.iinfo(noisy.dtype).max

NFFT, HOP = 4096, 1024
f, t, Sxx = spectrogram(noisy, fs=fs, nperseg=NFFT, noverlap=NFFT-HOP,
                        window="hann", detrend=False, mode="magnitude")

PERC = 25.0
frame_energy = compute_frame_energy(Sxx, power=True)
mask, thr = gap_mask_from_percentile(frame_energy, PERC)
print(f"{mask.sum()} frames ({mask.mean()*100:.1f}%) flagged as low-energy (thr={thr:.3e})")

noise_mag, noise_db = estimate_noise_from_gaps(Sxx, mask)

plt.figure(figsize=(8,3))
plt.plot(f, noise_mag, color = "green"); plt.title("Estimated noise spectrum (linear)", color = "red")
plt.xlabel("frequency (Hz)"); plt.ylabel("avg |X|"); plt.grid(True, alpha=0.3); plt.xlim(0, fs/2)
plt.show()

In [ ]:
plt.figure(figsize=(8,3))
plt.plot(f, noise_db, color = "green"); plt.title("Estimated noise spectrum (dB)", color = 'red')
plt.xlabel("frequency (Hz)"); plt.ylabel("dB"); plt.grid(True, alpha=0.3); plt.xlim(0, fs/2)
plt.show()

## Tonal noise suppression


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Tuple, Dict, List, Optional

from scipy.io import wavfile
from scipy.signal import (
    find_peaks, savgol_filter, iirnotch, sosfiltfilt, tf2sos
)

# -------------------- core helpers --------------------

def _odd_at_most(n: int) -> int:
    """Largest odd integer ≤ n, but at least 3."""
    n = int(n)
    if n < 3: return 3
    return n if (n % 2 == 1) else (n - 1)

def enhance_noise_spectrum_db(
    noise_mag: np.ndarray,
    *,
    sg_win: int = 31,
    sg_poly: int = 3,
    eps: float = 1e-12
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Convert |X|(f) -> dB, estimate a smooth baseline (Savitzky–Golay),
    and return enhancement: dB - baseline_dB.
    """
    noise_db = 20.0 * np.log10(np.maximum(noise_mag, eps))

    # Make SG params safe for the given length
    sg_win_safe = min(_odd_at_most(sg_win), len(noise_db) - (1 - (len(noise_db) % 2)))
    sg_win_safe = max(3, _odd_at_most(sg_win_safe))
    sg_poly_safe = min(sg_poly, sg_win_safe - 1)

    baseline_db = savgol_filter(noise_db, window_length=sg_win_safe,
                                polyorder=sg_poly_safe, mode="interp")
    enh_db = noise_db - baseline_db
    return noise_db, baseline_db, enh_db

def pick_tonal_peaks(
    f: np.ndarray,
    enh_db: np.ndarray,
    *,
    target_spacing_hz: float = 10.0,
    abs_prom_db: float = 3.0,
    top_k: int = 10
) -> Tuple[np.ndarray, np.ndarray, Dict[str, np.ndarray]]:
    """
    Peak picking on enhancement curve with prominence + min spacing in Hz.
    Returns (peak_bins, peak_freqs, props).
    """
    df = float(f[1] - f[0])
    min_dist_bins = max(1, int(round(target_spacing_hz / df)))

    peaks, props = find_peaks(enh_db, prominence=abs_prom_db, distance=min_dist_bins)
    if peaks.size:
        order = np.argsort(props["prominences"])[::-1][:top_k]
        peaks = peaks[order]
        for key in props:
            props[key] = props[key][order]
    peak_freqs = f[peaks] if peaks.size else np.array([], dtype=float)
    return peaks, peak_freqs, {"df": df, "min_dist_bins": min_dist_bins, **props}

def design_notch_chain_sos(
    peak_freqs: np.ndarray,
    fs: float,
    *,
    Q: float = 30.0,
    guard_hz: float = 30.0
) -> Optional[np.ndarray]:
    """
    Build a cascade of notch filters (SOS) for given peaks.
    Skips frequencies too close to DC/NYQ by 'guard_hz'.
    """
    valid = [f0 for f0 in peak_freqs if (guard_hz < f0 < (fs/2 - guard_hz))]
    if not valid:
        return None
    sos_list = []
    for f0 in valid:
        w0 = f0 / (fs / 2.0)             # normalized [0..1]
        b, a = iirnotch(w0, Q=Q)
        sos = tf2sos(b, a)               # stable SOS per notch
        sos_list.append(sos)
    return np.vstack(sos_list) if sos_list else None

def apply_sos_chain(x: np.ndarray, sos: np.ndarray) -> np.ndarray:
    """Apply SOS sections sequentially with zero-phase filtering."""
    y = x
    for sec in sos:
        y = sosfiltfilt(sec[np.newaxis, :], y)
    return y

# -------------------- plotting --------------------

def plot_enhancement_with_peaks(
    f: np.ndarray,
    enh_db: np.ndarray,
    peak_bins: np.ndarray,
    *,
    fs: float,
    title: str = "Tonal peak candidates (baseline-removed)",
) -> None:
    plt.figure(figsize=(9,3))
    plt.plot(f, enh_db, label="dB over smooth baseline")
    if peak_bins.size:
        plt.plot(f[peak_bins], enh_db[peak_bins], "rx", label="selected peaks")
    plt.title(title)
    plt.xlabel("frequency (Hz)"); plt.ylabel("excess (dB)")
    plt.xlim(0, fs/2)
    plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout()
    plt.show()

# ---- parameters (customize here) ----
TARGET_SPACING_HZ = 20.0
ABS_PROM_DB       = 3.0
PEAK_TOP_K        = 20
Q                 = 33.0

# 1) Enhancement
noise_db, baseline_db, enh_db = enhance_noise_spectrum_db(
    noise_mag, sg_win=31, sg_poly=3
)

# 2) Peak picking
peak_bins, peak_freqs, meta = pick_tonal_peaks(
    f, enh_db,
    target_spacing_hz=TARGET_SPACING_HZ,
    abs_prom_db=ABS_PROM_DB,
    top_k=PEAK_TOP_K
)

print(f"df = {meta['df']:.2f} Hz | min spacing = {meta['min_dist_bins']} bins "
      f"(~{meta['min_dist_bins']*meta['df']:.1f} Hz)")
print("Detected tonal peaks (Hz):", np.round(peak_freqs, 2).tolist())

# 3) Visualize
plot_enhancement_with_peaks(
    f, enh_db, peak_bins, fs=fs,
    title="Detected tonal components (after baseline removal)"
)

# 4) Design & apply notch chain
sos = design_notch_chain_sos(peak_freqs, fs, Q=Q, guard_hz=30.0)

if sos is not None:
    denoised_notch = apply_sos_chain(noisy, sos)
    print("Applied notch frequencies (Hz):", np.round(peak_freqs, 1).tolist())


## Gap selection for spectral subtraction


In [ ]:
# --- run once after you compute your spectrogram ---
PERC = 20.0   # lowest-energy 20% of frames
frame_energy = compute_frame_energy(Sxx, power=True)
gap_mask, thr = gap_mask_from_percentile(frame_energy, PERC)

print(f"{gap_mask.sum()} frames ({gap_mask.mean()*100:.1f}%) "
      f"flagged as gaps, threshold={thr:.3e}")

## Spectral subtraction and audio export


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Optional, Tuple
from scipy.signal import spectrogram, istft
from scipy.io import wavfile


# ------------------ helpers ------------------

def estimate_noise_power_from_gaps(
    Sxx_gap_mag: np.ndarray,
    gap_mask: np.ndarray
) -> np.ndarray:
    """
    Estimate per-frequency noise power from magnitude spectrogram Sxx (F x T).
    Uses mean(|X|^2) across time frames flagged as 'gaps'.
    If no gaps, falls back to global mean power.
    """
    if gap_mask.sum() == 0:
        return (Sxx_gap_mag ** 2).mean(axis=1)
    return (Sxx_gap_mag[:, gap_mask] ** 2).mean(axis=1)


def stft_complex(
    x: np.ndarray, fs: int, nfft: int, hop: int, window: str = "hann"
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Complex STFT via scipy.signal.spectrogram with mode='complex'.
    Returns (f, t, S) with S shape [F, T].
    """
    f, t, S = spectrogram(
        x, fs=fs, nperseg=nfft, noverlap=nfft - hop,
        window=window, detrend=False, mode="complex"
    )
    return f, t, S


def spectral_subtraction_mag(
    mag: np.ndarray,            # |S| (F x T)
    noise_pow: np.ndarray,      # noise power per freq (F,)
    *,
    beta: float = 1.2,          # oversubtraction factor
    floor_db: float = -20.0     # magnitude floor relative to |S|
) -> np.ndarray:
    """
    Classical spectral subtraction on magnitude spectrum:
        Xpow_est = max(|S|^2 - beta * noise_pow, 0)
        |S|_ss   = max(sqrt(Xpow_est), floor_lin * |S|)
    floor_db is in dB relative to |S| at each bin: floor_lin = 10^(floor_db/20).
    """
    eps = 1e-12
    Ypow = mag ** 2
    Xpow_est = np.maximum(Ypow - beta * noise_pow[:, None], 0.0)
    floor_lin = 10.0 ** (floor_db / 20.0)
    mag_ss = np.maximum(np.sqrt(Xpow_est), floor_lin * mag + eps)
    return mag_ss


def istft_from_mag_phase(
    mag: np.ndarray, phase: np.ndarray, fs: int, nfft: int, hop: int, window: str = "hann"
) -> np.ndarray:
    """Recompose complex STFT from mag/phase and return time-domain signal."""
    S_out = mag * np.exp(1j * phase)
    _, x = istft(S_out, fs=fs, nperseg=nfft, noverlap=nfft - hop, window=window)
    return x


def save_wav_int16(path: Path, fs: int, x: np.ndarray) -> None:
    """Clamp to [-1,1] and save as 16-bit PCM WAV."""
    x16 = (np.clip(x, -1.0, 1.0) * 32767.0).astype(np.int16)
    wavfile.write(str(path), fs, x16)


def mag_db_rfft(x: np.ndarray, fs: int, nfft: int = 8192) -> Tuple[np.ndarray, np.ndarray]:
    """One-sided magnitude spectrum in dB (rfft)."""
    X = np.fft.rfft(x, n=nfft)
    f = np.fft.rfftfreq(nfft, 1.0 / fs)
    db = 20.0 * np.log10(np.abs(X) + 1e-12)
    return f, db


def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.asarray(x, dtype=float) ** 2)))


# ------------------ main routine ------------------

def denoise_spectral_subtraction(
    noisy: np.ndarray,
    fs: int,
    *,
    # noise model inputs from a prior step:
    Sxx_gap_mag: np.ndarray,       # magnitude spectrogram used for gap detection (F x T)
    gap_mask: np.ndarray,          # boolean mask over time (T,)
    # processing parameters:
    nfft: int = 4096,
    hop: int = 1024,
    beta: float = 1.2,
    floor_db: float = -20.0,
    target: Optional[np.ndarray] = None,  # if provided (e.g., post-notch), denoise this
) -> Tuple[np.ndarray, np.ndarray]:
    """
    End-to-end: estimate noise power from gaps, STFT target, spectral subtraction, ISTFT.
    Returns (denoised_signal, noise_power_vector).
    """
    tgt = noisy if target is None else target

    # 1) Estimate noise power per frequency (from gap frames on the reference Sxx)
    noise_pow = estimate_noise_power_from_gaps(Sxx_gap_mag, gap_mask)

    # 2) STFT of target signal
    f2, t2, S = stft_complex(tgt, fs=fs, nfft=nfft, hop=hop, window="hann")
    mag = np.abs(S)
    phase = np.angle(S)

    # 3) Spectral subtraction (magnitude domain)
    mag_ss = spectral_subtraction_mag(mag, noise_pow, beta=beta, floor_db=floor_db)

    # 4) ISTFT and trim to original length
    denoised = istft_from_mag_phase(mag_ss, phase, fs=fs, nfft=nfft, hop=hop, window="hann")
    denoised = denoised[: len(tgt)]

    return denoised, noise_pow


# ------------------ example usage & plots ------------------

def run_and_report(
    noisy: np.ndarray,
    fs: int,
    *,
    Sxx_gap_mag: np.ndarray,
    gap_mask: np.ndarray,
    nfft: int = 2048,
    hop: int = 1024,
    target: Optional[np.ndarray] = None,
    beta: float = 1.2,
    floor_db: float = -20.0,
    out_path: Path = OUTPUT_DIR / "denoise1.wav",
    title_suffix: str = " (spectral subtraction)"
) -> None:
    """
    Runs denoising, saves WAV, plots spectra (noisy / optional target / final),
    and prints RMS values.
    """
    denoised, noise_pow = denoise_spectral_subtraction(
        noisy, fs, Sxx_gap_mag=Sxx_gap_mag, gap_mask=gap_mask,
        nfft=nfft, hop=hop, beta=beta, floor_db=floor_db, target=target
    )

    # Save
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    save_wav_int16(Path(out_path), fs, denoised)
    print("Saved:", out_path)

    # Spectra
    fN, dN = mag_db_rfft(noisy, fs)
    if target is not None:
        fI, dI = mag_db_rfft(target, fs)
    fD, dD = mag_db_rfft(denoised, fs)

    plt.figure(figsize=(10, 4))
    plt.plot(fN, dN, label="noisy")
    if target is not None:
        plt.plot(fI, dI, label="intermediate (e.g., notched)")
    plt.plot(fD, dD, label="final" + title_suffix)
    plt.title("One-sided magnitude spectra (dB) — before vs. after")
    plt.xlabel("frequency (Hz)")
    plt.ylabel("magnitude (dB)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.xlim(0, fs / 2)
    plt.tight_layout()
    plt.show()

    # RMS
    print("RMS noisy :", rms(noisy))
    if target is not None:
        print("RMS inter:", rms(target))
    print("RMS final :", rms(denoised))

run_and_report(
    noisy, fs,
    Sxx_gap_mag=Sxx, gap_mask=gap_mask,
    nfft=NFFT, hop=HOP,
    target=denoised_notch if 'denoised_notch' in globals() else None,
    beta=1.1, floor_db=-15.0,
    out_path=OUTPUT_DIR / "denoise1.wav",
    title_suffix=" (spectral subtraction)"
)